# Кластеризация клиентов (K-means) + 3D-визуализация
**Шаг 3 · кейс «GigaChat + Python»** — программа Сбер «Аналитика 360»

Сценарий встречи: код создаётся в GigaChat, копируется в IDE/ноутбук, запускается.
Этот ноутбук — эталонный результат такого диалога: сегментация клиентов K-means и
3D-график, на котором видно то, чего не видно в плоской таблице Excel.

**Бизнес-эффект:** точная сегментация → адресные коммуникации → экономия бюджета рассылок.

Промпт, с которого всё начинается (вставьте в GigaChat):
> Ты — аналитик данных. У меня CSV с клиентами: сумма покупок за год, число покупок,
> дней с последней покупки. Напиши код на Python: загрузка pandas, нормализация,
> кластеризация K-means на 4 сегмента, подбор числа кластеров методом локтя,
> 3D-scatter (matplotlib) с раскраской по сегментам, таблица «сегмент → средние значения →
> рекомендация по коммуникации». Прокомментируй каждый шаг для руководителя.

In [ ]:
import numpy as np
import pandas as pd

# Синтетические клиенты банка: monetary (тыс. руб/год), frequency (покупок/год), recency (дней)
rng = np.random.default_rng(42)
segments = [
    dict(n=300, monetary=(900, 180), frequency=(48, 10), recency=(12, 6)),    # активные премиум
    dict(n=500, monetary=(320, 90),  frequency=(22, 6),  recency=(25, 10)),   # ядро
    dict(n=400, monetary=(140, 50),  frequency=(8, 3),   recency=(90, 25)),   # затухающие
    dict(n=250, monetary=(60, 25),   frequency=(3, 1.5), recency=(210, 40)),  # спящие
]
parts = []
for s in segments:
    parts.append(pd.DataFrame({
        "monetary":  rng.normal(*s["monetary"],  s["n"]).clip(5),
        "frequency": rng.normal(*s["frequency"], s["n"]).clip(1),
        "recency":   rng.normal(*s["recency"],   s["n"]).clip(1),
    }))
df = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)
df.describe().round(1)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

X = StandardScaler().fit_transform(df)

# Метод локтя: где перестаёт заметно падать инерция — там разумное число кластеров
inertia = {k: KMeans(n_clusters=k, n_init=10, random_state=0).fit(X).inertia_ for k in range(2, 9)}
plt.figure(figsize=(6, 3))
plt.plot(list(inertia), list(inertia.values()), marker="o")
plt.xlabel("Число кластеров k"); plt.ylabel("Инерция"); plt.title("Метод локтя")
plt.show()

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=0)
df["segment"] = km.fit_predict(X)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
sc = ax.scatter(df.monetary, df.frequency, df.recency, c=df.segment, cmap="tab10", s=14, alpha=0.7)
ax.set_xlabel("Сумма покупок, тыс. ₽/год"); ax.set_ylabel("Покупок/год"); ax.set_zlabel("Дней с покупки")
ax.set_title("Сегменты клиентов (K-means, 3D)")
plt.show()

In [ ]:
summary = df.groupby("segment").agg(
    клиентов=("monetary", "size"),
    сумма_тыс=("monetary", "mean"),
    покупок=("frequency", "mean"),
    дней_с_покупки=("recency", "mean"),
).round(1).sort_values("сумма_тыс", ascending=False)

reco = ["Персональный менеджер, премиум-предложения",
        "Кросс-продажи, программы лояльности",
        "Реактивация: индивидуальный триггер",
        "Дешёвые каналы или пауза в коммуникациях"]
summary["рекомендация"] = reco[: len(summary)]
summary

## О чём это руководителю
- Плоская таблица «средних по больнице» скрывает 4 разных поведения — 3D-график делает их видимыми.
- Каждому сегменту — свой канал и бюджет коммуникаций: это и есть измеримый эффект сегментации.
- Код не писался вручную — его сгенерировал ассистент; роль руководителя — правильный вопрос и проверка вывода (Supervisor-контроль).

**Проверьте себя (типичные ошибки Шага 3):** число кластеров обосновано (метод локтя), а не «красиво»;
сегменты устойчивы на новом периоде данных; интерпретация — гипотеза до проверки, а не факт.